In [1]:
import numpy as np
import cupy as cp
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import sys
from google.colab import files

# ═══════════════════════════════════════════════════════════════
# FUNCIONS
# ═══════════════════════════════════════════════════════════════
def geoeq(swarm):
    """
    Calcula la derivada de l'eixam de fotons respecte al paràmetre afí,
    és a dir, el membre dret de l'equació geodèsica:

        d²xᵘ/dλ² = -Γᵘₐᵦ (dxᵃ/dλ)(dxᵝ/dλ)

    L'array swarm té forma (Ny, Nz, 8), on els quatre primers components
    són les coordenades (t, r, θ, φ) i els quatre últims són les seves
    derivades respecte a λ. La derivada de les coordenades és simplement
    la 4-velocitat, i la derivada de la 4-velocitat és la suma dels
    símbols de Christoffel. Els fotons marcats a stopped_mask no
    s'actualitzen.
    """
    dswarm = cp.zeros_like(swarm)
    dswarm[..., :4] = swarm[..., 4:]

    for idx1 in range(4):
        for idx2 in range(4):
            for idx3 in range(idx2, 4):
                gamma = christoffel(swarm, (idx1, idx2, idx3))
                if gamma is not None:
                    # Aprofitem la simetria Γᵘₐᵦ = Γᵘᵦₐ: els termes amb
                    # índexs creuats (a≠b) apareixen dues vegades a la suma.
                    dswarm[..., idx1 + 4] -= (2 - (idx2 == idx3)) * gamma * swarm[..., idx2 + 4] * swarm[..., idx3 + 4]

    # Els fotons que han caigut o escapat no s'evolucionen més.
    dswarm[stopped_mask != 0] = 0.0
    return dswarm


def update_masks(swarm, new_swarm):
    """
    Comprova la posició de cada fotó i actualitza les màscares globals
    stopped_mask i canvas.

    Un fotó es marca com a aturat (stopped_mask = True) si:
      - Ha caigut dins l'horitzó de successos (r ≤ 1.1 * r_horizon), o
      - Ha escapt massa lluny (r > 60).

    Un fotó s'afegeix al canvas si en el pas actual ha creuat el pla
    equatorial (canvi de signe de cos(θ)) i es troba dins del disc
    d'acreció (r_ISCO < r < r_disk). En aquest cas s'acumula el valor
    del radi, de manera que les zones amb més creuaments surten més
    brillants a la imatge final.
    """
    _, r, th, _, _, _, _, _   = swarm.transpose(2, 0, 1)
    _, _, n_th, _, _, _, _, _ = new_swarm.transpose(2, 0, 1)

    fell_into_horizon   = (r <= 1.1 * r_horizon)
    went_too_far        = (r > 60)
    crossed_the_equator = (cp.cos(th) * cp.cos(n_th) < 0)
    hit_the_disk        = crossed_the_equator & (r > r_ISCO) & (r < r_disk)

    canvas[hit_the_disk] += r[hit_the_disk]
    stopped_mask[fell_into_horizon | went_too_far] = True


def RK4(lam, dlam, swarm):
    """
    Avança l'eixam de fotons un pas de mida dlam en el paràmetre afí,
    utilitzant el mètode de Runge-Kutta d'ordre 4.

    El mètode avalua la derivada (geoeq) en quatre punts estratègics
    de l'interval i en fa una mitjana ponderada:

        k1 = f(λ,       y)
        k2 = f(λ+h/2,   y + h/2·k1)
        k3 = f(λ+h/2,   y + h/2·k2)
        k4 = f(λ+h,     y + h·k3)

        y_{n+1} = y_n + h/6·(k1 + 2k2 + 2k3 + k4)

    Després d'actualitzar l'eixam, crida update_masks per registrar
    els creuaments del disc i els fotons aturats.
    """
    k1 = geoeq(swarm)
    k2 = geoeq(swarm + 0.5 * k1 * dlam)
    k3 = geoeq(swarm + 0.5 * k2 * dlam)
    k4 = geoeq(swarm + k3 * dlam)

    new_lam   = lam + dlam
    new_swarm = swarm + dlam * (k1 + 2*k2 + 2*k3 + k4) / 6

    update_masks(swarm, new_swarm)
    return new_lam, new_swarm


def toCartesian(swarm):
    """
    Converteix les coordenades de Schwarzschild (t, r, θ, φ) a
    coordenades cartesianes (x, y, z) mitjançant les relacions:

        x = r·sin(θ)·cos(φ)
        y = r·sin(θ)·sin(φ)
        z = r·cos(θ)

    S'utilitza per representar les trajectòries dels fotons en
    matplotlib.
    """
    _, r, th, ph, _, _, _, _ = swarm.transpose(2, 0, 1)
    x = r * cp.sin(th) * cp.cos(ph)
    y = r * cp.sin(th) * cp.sin(ph)
    z = r * cp.cos(th)
    return x, y, z


def christoffel(swarm, indices):
    """
    Retorna el símbol de Christoffel Γᵘₐᵦ de la mètrica de Schwarzschild
    per als índexs donats, avaluat a cada punt de l'eixam.

    Els símbols no nuls de la mètrica de Schwarzschild en coordenades
    (t, r, θ, φ) són:

        Γ^t_tr = Γ^t_rt = M / (r(r−2M))
        Γ^r_tt = M(r−2M) / r³
        Γ^r_rr = −M / (r(r−2M))
        Γ^r_θθ = −(r−2M)
        Γ^r_φφ = −(r−2M)·sin²(θ)
        Γ^θ_rθ = Γ^θ_θr = 1/r
        Γ^θ_φφ = −sin(θ)·cos(θ)
        Γ^φ_rφ = Γ^φ_φr = 1/r
        Γ^φ_θφ = Γ^φ_φθ = cos(θ)/sin(θ)

    Retorna None per a qualsevol combinació d'índexs que correspongui
    a un símbol nul, de manera que geoeq pot saltar-se el càlcul.
    El resultat té la forma (Ny, Nz), igual que r i th.
    """
    _, r, th, ph, _, _, _, _ = swarm.transpose(2, 0, 1)
    sin_th = cp.sin(th)

    if indices == (1, 0, 0):
        gamma = M * (r - 2 * M) / r**3
    elif indices in ((0, 0, 1), (0, 1, 0)):
        gamma = M / (r * (r - 2 * M))
    elif indices == (1, 1, 1):
        gamma = -M / (r * (r - 2 * M))
    elif indices == (1, 2, 2):
        gamma = -(r - 2 * M)
    elif indices == (1, 3, 3):
        gamma = -(r - 2 * M) * sin_th**2
    elif indices in ((2, 1, 2), (2, 2, 1)):
        gamma = 1 / r
    elif indices == (2, 3, 3):
        gamma = -sin_th * cp.cos(th)
    elif indices in ((3, 1, 3), (3, 3, 1)):
        gamma = 1 / r
    elif indices in ((3, 2, 3), (3, 3, 2)):
        gamma = cp.cos(th) / sin_th
    else:
        gamma = None

    return gamma





# ═══════════════════════════════════════════════════════════════
# PARÀMETRES DEL SISTEMA
# ═══════════════════════════════════════════════════════════════

shape   = (5*400, 5*350)  # Resolució de la pantalla (Ny, Nz) en píxels.
y_range = (-25, 25)       # Rang de la pantalla en la direcció azimutal.
z_range = (-20, 20)       # Rang de la pantalla en la direcció polar.
M       = 1               # Massa del forat negre en unitats geomètriques.

# Radis característics de la geometria de Schwarzschild.
r_horizon = 2 * M              # Horitzó de successos.
r_shadow  = 3 * np.sqrt(3) * M # Radi de la silueta (paràmetre d'impacte crític).
r_ISCO    = 6 * M              # Òrbita circular estable més interna (ISCO).
r_disk    = 18                 # Radi exterior del disc d'acreció.

# ═══════════════════════════════════════════════════════════════
# BUCLE PRINCIPAL: IMATGE PER A CADA ANGLE DE VISIÓ
# ═══════════════════════════════════════════════════════════════
# Per a cada angle, la càmera es col·loca a una distància u_screen
# del forat negre i la pantalla es rota en el pla (x, z). Un angle
# zero correspon a una vista de costat (disc de cantell) i π/2
# correspon a una vista des de dalt (disc com un anell).

view_angle_vals = [0.00, 0.25, 0.52, 0.79, 1.05, 1.22, 1.40, np.pi/2]

for i in range(len(view_angle_vals)):

  view_angle = view_angle_vals[i]

  # ── Condicions inicials de la pantalla ──────────────────────
  # La pantalla es defineix en coordenades cartesianes (x, y, z).
  # u_screen és la distància de la càmera al forat negre, i v_screen
  # és l'eix de la pantalla perpendicular al pla equatorial.
  # La rotació de view_angle inclina la càmera respecte al disc:
  #   x_screen = cos(α)·u − sin(α)·v
  #   z_screen = cos(α)·v + sin(α)·u
  t_screen = 0.0
  u_screen = 50.0
  y_screen = cp.linspace(*y_range, shape[0])[:, None]
  v_screen = cp.linspace(*z_range, shape[1])[None, :]

  x_screen = cp.cos(view_angle) * u_screen - cp.sin(view_angle) * v_screen
  z_screen = cp.cos(view_angle) * v_screen + cp.sin(view_angle) * u_screen

  # ── Pas a coordenades esfèriques de Schwarzschild ───────────
  t  = t_screen
  r  = cp.sqrt(x_screen**2 + y_screen**2 + z_screen**2)
  th = cp.arccos(z_screen / r)
  ph = cp.arctan2(y_screen, x_screen)

  # ── 4-velocitat inicial ─────────────────────────────────────
  # Els fotons han de viatjar en la direcció −x rotada per view_angle.
  # ddx i ddz són les components esfèriques dels vectors unitaris
  # cartesians d/dx i d/dz, respectivament.
  ddx = ( cp.sin(th) * cp.cos(ph),
          cp.cos(th) * cp.cos(ph) / r,
         -cp.sin(ph) / (r * cp.sin(th) + 1e-5 * (cp.sin(th) == 0)))
  ddz = ( cp.cos(th), -cp.sin(th) / r, 0*r)

  dr_dl  = -cp.cos(view_angle) * ddx[0] - cp.sin(view_angle) * ddz[0]
  dth_dl = -cp.cos(view_angle) * ddx[1] - cp.sin(view_angle) * ddz[1]
  dph_dl = -cp.cos(view_angle) * ddx[2] - cp.sin(view_angle) * ddz[2]

  # Normalitzem perquè dt/dλ = 1 i la 4-velocitat sigui nul·la (gᵤᵥkᵘkᵛ = 0).
  norm2  = dr_dl**2 / (1 - 2*M/r) + r**2 * (dth_dl**2 + cp.sin(th)**2 * dph_dl**2)
  factor = cp.sqrt(norm2 / (1 - 2*M/r))

  # ── Inicialització de l'eixam ────────────────────────────────
  # Cada píxel (i, j) correspon a un fotó. swarm_init té forma
  # (Ny, Nz, 8): els quatre primers components són (t, r, θ, φ)
  # i els quatre últims són les seves derivades respecte a λ.
  swarm_init = cp.zeros(shape + (8,))
  swarm_init[..., 0] = t
  swarm_init[..., 1] = r
  swarm_init[..., 2] = th
  swarm_init[..., 3] = ph
  swarm_init[..., 4] = 1
  swarm_init[..., 5] = dr_dl  / factor
  swarm_init[..., 6] = dth_dl / factor
  swarm_init[..., 7] = dph_dl / factor

  # Comprovació: la 4-velocitat ha de ser nul·la.
  if not cp.allclose(
      -(1 - 2*M/r) * swarm_init[..., 4]**2
      + swarm_init[..., 5]**2 / (1 - 2*M/r)
      + r**2 * (swarm_init[..., 6]**2 + cp.sin(th)**2 * swarm_init[..., 7]**2), 0):
    sys.exit

  # Màscara per identificar la zona problemàtica (fotons prop de l'eix z),
  # que necessitarà ser reintegrada amb un pas més petit.
  slow_mask = (cp.abs(y_screen) < 0.5) & cp.ones(r.shape, dtype=bool)

  # ── Integració global ────────────────────────────────────────
  # Primera passada: integrem tots els fotons amb un pas estàndard.
  lam, dlam = 0, 0.1
  steps = 2500
  dif   = 10

  swarm        = swarm_init.copy()
  stopped_mask = cp.zeros_like(r, dtype=bool)
  canvas       = cp.zeros_like(r)

  for it in tqdm(range(steps)):
    lam, swarm = RK4(lam, dlam, swarm)

  image = canvas.copy()
  print(f'Paràmetre afí al final de la integració: {lam:.2f}')

  # ── Reintegració de la zona problemàtica ────────────────────
  # Els fotons prop de l'eix z pateixen inestabilitats numèriques
  # amb el pas estàndard. Es tornen a integrar des del principi
  # amb un pas dif vegades més petit per garantir la precisió.
  coords = cp.where(slow_mask)
  y_min, y_max = int(coords[0].min()), int(coords[0].max())
  z_min, z_max = int(coords[1].min()), int(coords[1].max())

  swarm_c      = swarm_init[y_min:y_max+1, z_min:z_max+1, :].copy()
  steps_c      = dif * steps
  lam          = 0
  stopped_mask = cp.zeros_like(swarm_c[..., 0], dtype=bool)
  canvas       = cp.zeros_like(swarm_c[..., 1])

  for it in tqdm(range(steps_c)):
    lam, swarm_c = RK4(lam, dlam/dif, swarm_c)

  print(f'Paràmetre afí al final de la integració 2: {lam:.2f}')

  # Substituïm la zona problemàtica a la imatge final.
  image[y_min:y_max+1, z_min:z_max+1] = canvas

  # ── Representació gràfica ────────────────────────────────────
  contour  = cp.linspace(0, 2*cp.pi, 100)
  contour1 = cp.asnumpy(contour)
  r1, r2, r3 = r_shadow, r_horizon, r_ISCO

  # Imatge sense anotacions.
  fig, ax = plt.subplots(1, 1, figsize=(9, 6))
  plt.rc('font', size=16)
  plt.imshow(cp.asnumpy(image).T, cmap='hot', interpolation='none',
             origin='lower', extent=y_range + z_range)
  plt.title(f"view angle = {view_angle:.2f} rad")
  plt.tight_layout()
  plt.savefig(f"BH_{view_angle*100:.0f}.png")

  # Imatge amb cercles de referència (shadow, horitzó, ISCO).
  fig, ax = plt.subplots(1, 1, figsize=(9, 6))
  plt.rc('font', size=16)
  plt.imshow(cp.asnumpy(image).T, cmap='hot', interpolation='none',
             origin='lower', extent=y_range + z_range)
  plt.title(f"view angle = {view_angle:.2f} rad")
  ax.plot(r1 * np.cos(contour1), r1 * np.sin(contour1), '--w', linewidth=1, label=f'shadow  r={r1:.2f}M')
  ax.plot(r2 * np.cos(contour1), r2 * np.sin(contour1), '--w', linewidth=1, label=f'horitzó r={r2}M')
  ax.plot(r3 * np.cos(contour1), r3 * np.sin(contour1), '--w', linewidth=1, label=f'ISCO    r={r3}M')
  ax.legend(fontsize=11)
  plt.tight_layout()
  plt.savefig(f"BH_{view_angle*100:.0f}_circles.png")

ModuleNotFoundError: No module named 'cupy'